## Text input

https://platform.openai.com/docs/models

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import create_agent

agent = create_agent(
    model='gpt-5-nano',
    system_prompt="You are a helpful assistant that describes images accurately. If something is unclear, say so instead of guessing.",
)

In [3]:
from langchain.messages import HumanMessage

question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

There isn’t one. The Moon has no government or political status, so it has no capital city. If you’re thinking of a fictional universe, tell me which one and I can name its capital as that setting describes. I can also share real lunar features (like Mare Tranquillitatis or Tycho crater) if that helps.


## Image input

In [4]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [5]:
print(uploader.value)

({'name': 'moon.png', 'type': 'image/png', 'size': 358916, 'content': <memory at 0x125c8b400>, 'last_modified': datetime.datetime(2026, 11, 17, 18, 48, 17, 994000, tzinfo=datetime.timezone.utc)},)


In [6]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

print(uploaded_file)

# This is a memoryview
content_mv = uploaded_file["content"]

print(content_mv)

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

print(img_bytes)

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")
print(img_b64)

{'name': 'moon.png', 'type': 'image/png', 'size': 358916, 'content': <memory at 0x125c8b400>, 'last_modified': datetime.datetime(2026, 11, 17, 18, 48, 17, 994000, tzinfo=datetime.timezone.utc)}
b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x02\x1c\x00\x00\x01h\x08\x02\x00\x00\x00\x93\xc7\xb5a\x00\x00\x00\x01sRGB\x00\xae\xce\x1c\xe9\x00\x00\x00\xa6eXIfMM\x00*\x00\x00\x00\x08\x00\x06\x01\x12\x00\x03\x00\x00\x00\x01\x00\x01\x00\x00\x01\x1a\x00\x05\x00\x00\x00\x01\x00\x00\x00V\x01\x1b\x00\x05\x00\x00\x00\x01\x00\x00\x00^\x01(\x00\x03\x00\x00\x00\x01\x00\x02\x00\x00\x011\x00\x02\x00\x00\x00\x15\x00\x00\x00f\x87i\x00\x04\x00\x00\x00\x01\x00\x00\x00|\x00\x00\x00\x00\x00\x00\x01,\x00\x00\x00\x01\x00\x00\x01,\x00\x00\x00\x01Adobe Stock Platform\x00\x00\x00\x03\xa0\x01\x00\x03\x00\x00\x00\x01\x00\x01\x00\x00\xa0\x02\x00\x04\x00\x00\x00\x01\x00\x00\x02\x1c\xa0\x03\x00\x04\x00\x00\x00\x01\x00\x00\x01h\x00\x00\x00\x00\x81\x11C\x8b\x00\x00\x00\tpHYs\x00\x00.#\x00\x00.#\x01x\xa5?v\x00\x00\x03\xbdiTXt

In [7]:
from langchain.messages import HumanMessage


multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this picture"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

This is a science-fiction landscape showing an otherworldly world.

What’s in the scene
- A massive, pale blue planet (or large moon) dominates the left side of the sky, giving a sense of being far from Earth.
- In the mid-ground, a futuristic city rises among jagged, mountainous rock formations. The skyline features tall, sleek towers with glowing blue accents.
- The foreground and middle ground are a rugged, desert-like terrain with rocky outcrops and spires.
- A couple of glowing blue pools or circles on the ground add an eerie, alien glow.

Mood and atmosphere
- Cool blue lighting gives the scene a calm, otherworldly feel, with a mix of desolation and technological wonder.
- The contrast between the ancient, rugged foreground and the advanced city suggests a world where exploration and civilization intersect.

Scale and composition
- The large planet/moon on the left provides a strong sense of scale and depth.
- The city is distant but prominent, while the jagged rocks in the foreg

## Audio input

In [16]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

ModuleNotFoundError: No module named 'tqdm'

In [8]:
agent = create_agent(
    model='gpt-audio',
)

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this audio file"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

NameError: name 'aud_b64' is not defined